# 06 - Build SMS Accident Samples

**目的：** 把交控事故簡訊（SMS）轉成 VD 速度矩陣與特徵，供 `dispatch + SMS` 資料軸實驗使用。

SMS 來源涵蓋全國路網且與派遣資料重疊（2024-04 ~ 2025-05）。依專案決策，**重疊期間以派遣資料為準、忽略 SMS**；SMS 只用於派遣資料涵蓋範圍之外的期間（預設報案日 < 2024-04-01，即 2023-06 ~ 2024-03）。

## 流程
1. `process_sms_data.py` — 解析 24 個月 xlsx → dispatch 相容 CSV（`data/accident_data_sms.csv`）。
2. `build_accident_samples.py --tag sms` — 從 `vd_station_5m` 切出 SMS 事故矩陣（`data/accident_samples_5m_sms.npz`）。
3. `build_incident_features.py` — 建 SMS 特徵（availability schema）。
4. Sanity check — coverage 分布 + 抽樣 heatmap。

**前置需求：** `vd_station_5m` 需含 2023-06 ~ 2024-03 的站台資料（已建好）。

In [ ]:
import os, sys

# ── environment detection ─────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    _DEFAULT_ROOT = '/content/drive/MyDrive/陽交大高速公路計畫/main'
else:
    _here = os.path.abspath(os.getcwd())
    _DEFAULT_ROOT = os.path.dirname(_here) if os.path.basename(_here) == 'notebooks' else _here

PROJECT_ROOT = os.environ.get('PROJECT_ROOT', _DEFAULT_ROOT)
%cd {PROJECT_ROOT}
print('PROJECT_ROOT =', PROJECT_ROOT)

---
## 1. 解析 SMS → dispatch 相容 CSV

輸出 `data/accident_data_sms.csv`，欄位名稱與 `accident_data.csv` 相同，故下游 sample / feature 腳本可直接吃。

- 民國/西元年混用已正規化；里程 `158.5公里` 之類單位後綴會被剝除。
- 車道數由 `data/geometry` 幾何資料以里程查得 → `主線車道數`；阻斷主線數由 SMS 車道欄推得 → `剩餘車道數`。
- 預設只保留報案日 < 2024-04-01 且落在 VD 涵蓋里程內的記錄（約 4,479 筆）。

In [ ]:
!uv run scripts/data/process_sms_data.py

---
## 2. 建 SMS 事故矩陣 NPZ

`--tag sms` 讓所有輸出加上 `_sms` 後綴，避免覆蓋派遣資料的鎖版產物。`--no-csv` 避免在 Drive 產生數千個小 CSV（per-accident debug CSV 非必要）：
- `data/accident_samples_5m_sms.npz`
- `data/accident_samples_5m_sms_metadata.parquet`
- `data/metadata/accident_samples_report_sms.json`

In [ ]:
!uv run scripts/data/build_accident_samples.py \
    --accident-csv data/accident_data_sms.csv \
    --tag sms \
    --start-date 2023-06-01 --end-date 2024-03-31 \
    --no-csv --force

---
## 3. 建 SMS 特徵（availability schema）

輸出 `output/feature_engineering/incident_features_sms_44.csv`（33 value + 11 `avail_*`）。
SMS 多數 `事故類別` / `受傷` / `死亡` 欄為空 → 對應 `avail_*` = 0（unknown），不會被當成真實 0。

In [ ]:
!uv run scripts/features/build_incident_features.py \n    --accident-csv data/accident_data_sms.csv \n    --output-name incident_features_sms_44.csv

---
## 4. Sanity check

確認矩陣 shape、coverage 分布、feature 對齊，並抽一筆畫 observed heatmap。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path(PROJECT_ROOT)))

npz = np.load('data/accident_samples_5m_sms.npz', allow_pickle=False)
meta = pd.read_parquet('data/accident_samples_5m_sms_metadata.parquet')
feat = pd.read_csv('output/feature_engineering/incident_features_sms_44.csv', dtype={'task_id': str})

print('SMS matrix:', npz['matrix'].shape)
print('SMS metadata rows:', len(meta), '| status:', meta['status'].value_counts().to_dict())
print('filled_ratio >= 0.8:', int((meta['filled_ratio'] >= 0.8).sum()), '/', len(meta))
print('SMS feature rows:', len(feat), '| cols:', feat.shape[1])
ids_npz = set(npz['accident_ids'].astype(str)); ids_feat = set(feat['task_id'])
print('NPZ ids covered by features:', len(ids_npz & ids_feat), '/', len(ids_npz))

In [ ]:
# Coverage histogram + one observed heatmap
import matplotlib.pyplot as plt
from scripts.lib.windows import SampleWindow, subset
from scripts.lib import visualization as viz

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(meta['filled_ratio'], bins=30, color='#2980b9')
ax.axvline(0.8, color='red', ls='--', label='0.8 keep threshold')
ax.set_title('SMS sample filled_ratio'); ax.set_xlabel('filled_ratio'); ax.legend()
plt.show()

best = meta.sort_values('filled_ratio', ascending=False).iloc[0]
idx = int(np.where(npz['accident_ids'].astype(str) == str(best['accident_id']))[0][0])
full_w = SampleWindow(upstream_km=int(npz['window_upstream_km']), downstream_km=int(npz['window_downstream_km']),
                      time_before_min=int(npz['window_time_before_min']), time_after_min=int(npz['window_time_after_min']),
                      spatial_resolution_km=int(npz['window_spatial_resolution_km']), resolution_min=int(npz['window_resolution_min']))
m, w = subset(npz['matrix'][idx], full_w, upstream_km=25, downstream_km=10, time_before_min=30, time_after_min=145)
print('accident', best['accident_id'], best['freeway'], best['direction'], best['mileage_km'], 'km | filled', round(float(best['filled_ratio']), 3))
viz.heatmap(m, space_axis_km=w.space_offsets_m / 1000.0, time_offsets_min=w.time_offsets_min,
            title=f"SMS {best['accident_id']} observed", style=viz.STYLE_NOTEBOOK)
plt.show()